# 01 — Data

Anthropics [Economic Index](https://huggingface.co/datasets/Anthropic/EconomicIndex) samples Claude conversations, maps each one to an [O\*NET](https://www.onetonline.org/) occupational task, and classifies the collaboration mode: did the human use AI as an autonomous executor (*directive*), or as a collaborator (*task iteration*, *validation*, *learning*)?

Four releases span March 2025 to March 2026. The schema expanded substantially in January 2026, adding per-task continuous measures (AI autonomy, education estimates, success rates). This notebook downloads the data and constructs the task-level dataset used in the analysis.

In [1]:
import sys, os
sys.path.insert(0, "..")
os.chdir('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import (
    download_all, load_unified_release,
    build_task_feature_matrix, RELEASE_FILES,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 150

In [2]:
paths = download_all()
print(f'{len(paths)} files downloaded')

19 files downloaded


## Release structure

The four releases differ in what they measure. Release 1 (March 2025) provides only collaboration mode shares per task — five numbers per task, summing to 1. Releases 3--4 (January and March 2026) add nine additional facets per task, including continuous measures of AI autonomy and estimated human/AI education years.

We primarily use the **March 2026 release** for the cross-sectional task analysis.

In [3]:
release_files = [
    ('Mar 2025', 'data/release_2025_03_27_automation_vs_augmentation_by_task.csv'),
    ('Sep 2025', 'data/release_2025_09_15_data_intermediate_aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv'),
    ('Jan 2026', 'data/release_2026_01_15_data_intermediate_aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv'),
    ('Mar 2026', 'data/release_2026_03_24_data_aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv'),
]

for label, path in release_files:
    df = pd.read_csv(path)
    if 'facet' in df.columns:
        n_facets = df['facet'].nunique()
        n_tasks = df[df['facet'].str.fullmatch('onet_task', na=False)]['cluster_name'].nunique()
        print(f'{label:10s}  {len(df):>8,} rows   {n_facets:>2} facets   {n_tasks:>5} tasks')
    else:
        print(f'{label:10s}  {len(df):>8,} rows    5 modes    {len(df):>5} tasks')

Mar 2025       3,364 rows    5 modes     3364 tasks
Sep 2025     100,062 rows    7 facets    2618 tasks
Jan 2026     458,778 rows   34 facets    3170 tasks
Mar 2026     425,257 rows   34 facets    3260 tasks


## Collaboration modes

Each conversation is classified into one of five modes. The key distinction is *directive* — where the human hands off the task entirely — versus modes where the human stays in the loop.

| Mode | The human... |
|------|--------------|
| **Directive** | gives instructions, accepts output without significant revision |
| **Feedback loop** | iterates with AI, but AI drives the process |
| **Task iteration** | iterates with AI, with human driving |
| **Validation** | checks or verifies AI output |
| **Learning** | uses AI to understand something |

Anthropics groups the first two as *automation* and the last three as *augmentation*. We define **automation share** per task as the fraction of that task's conversations classified as directive or feedback loop.

In [4]:
# Global mode breakdown for March 2026
claude = load_unified_release('2026_03', platform='claude_ai')
api = load_unified_release('2026_03', platform='api')

def global_modes(df):
    m = (df['facet'] == 'collaboration') & (df['geography'] == 'global') & df['variable'].str.endswith('pct')
    return df[m].set_index('cluster_name')['value']

modes_order = ['directive', 'feedback loop', 'task iteration', 'validation', 'learning']
comp = pd.DataFrame({'Claude.ai': global_modes(claude), 'API': global_modes(api)}).reindex(modes_order)

print('Platform-level collaboration mode shares (%, March 2026):\n')
print(comp.to_string(float_format='{:.1f}'.format))

Platform-level collaboration mode shares (%, March 2026):

                Claude.ai  API
cluster_name                  
directive            32.6 58.2
feedback loop        11.5  9.4
task iteration       25.6  9.3
validation            4.9  3.4
learning             22.4  4.4


## Constructing the task dataset

From the March 2026 release, we extract one row per O\*NET task with: collaboration mode shares, AI autonomy score (1--5), human and AI education year estimates, task success rate, and conversation count. We then join to O\*NET's task-to-occupation mapping to attach SOC codes.

In [5]:
tasks = build_task_feature_matrix()
tu = tasks.drop_duplicates(subset='task_name').reset_index(drop=True)

print(f'{len(tu):,} unique tasks, mapped to {tasks["soc_code"].nunique()} occupations')
print()
for c in ['automation_share', 'ai_autonomy_mean', 'human_education_years',
          'success_rate', 'conversation_count']:
    n = tu[c].notna().sum()
    med = tu[c].median()
    print(f'  {c:28s}  {n:>5,}/{len(tu):,}  median = {med:.2f}')

3,259 unique tasks, mapped to 566 occupations

  automation_share              3,259/3,259  median = 0.22
  ai_autonomy_mean              3,259/3,259  median = 3.41
  human_education_years         3,259/3,259  median = 12.25
  success_rate                  2,739/3,259  median = 0.74
  conversation_count            3,259/3,259  median = 52.00
